# Pipeline d'extraction vidéo — Game Changer (phase 1)

Fait tourner exactement le même code que testé en local (calibration terrain, détection/tracking
football, ré-identification par embedding, regroupement global) sur GPU.

**Avant de lancer quoi que ce soit :**
1. `Exécution > Modifier le type d'exécution` → GPU (normalement déjà proposé par défaut ici).
2. Uploade ta vidéo dans ton Drive (ex. `Mon Drive/football/ALDM - FCSN Veo .mp4`) — lance l'upload
   maintenant, il continue en tâche de fond pendant que les cellules suivantes s'exécutent.
3. Modifie `VIDEO_PATH` dans la cellule ci-dessous pour pointer vers ce fichier.

In [ ]:
import torch
print("GPU disponible :", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))
else:
    print("Aucun GPU détecté — va dans Exécution > Modifier le type d'exécution et choisis GPU, "
          "puis relance cette cellule.")


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## Chemins — à adapter

`VIDEO_PATH` : où tu as uploadé la vidéo dans ton Drive.
`OUT_JSON` / `OUT_OVERLAY` : où les résultats seront sauvegardés dans ton Drive (survivent à la
fermeture de cette session Colab, contrairement au reste qui est effacé).

In [ ]:
VIDEO_PATH = "/content/drive/MyDrive/football/ALDM - FCSN Veo .mp4"
OUT_JSON = "/content/drive/MyDrive/football/resultat.json"
OUT_OVERLAY = "/content/drive/MyDrive/football/apercu_debug.mp4"
# Sur Drive (pas sur le disque local de la session, effacé à la déconnexion) — si la session est
# coupée, réexécute simplement la cellule du match complet plus bas : --resume reprend automatiquement
# depuis ce fichier s'il existe déjà, sinon repart de zéro sans rien casser.
CHECKPOINT = "/content/drive/MyDrive/football/checkpoint.pkl"


## Installation — dépendances, PnLCalib, poids

In [ ]:
# torch/torchvision volontairement absents de cette liste : Colab les fournit déjà, compilés pour
# le GPU de cette session — les réinstaller via pip risque de casser cette compatibilité CUDA.
!pip install -q ultralytics supervision trackers torchreid gdown opencv-python av easyocr scipy shapely pyyaml pillow tqdm


In [ ]:
import os
os.makedirs("vendor", exist_ok=True)
if not os.path.isdir("vendor/PnLCalib"):
    !git clone --depth 1 https://github.com/mguti97/PnLCalib.git vendor/PnLCalib

os.makedirs("vendor/PnLCalib/weights", exist_ok=True)
for f in ["SV_kp", "SV_lines"]:
    path = f"vendor/PnLCalib/weights/{f}"
    if not os.path.isfile(path):
        !curl -fL -o "{path}" "https://github.com/mguti97/PnLCalib/releases/download/v1.0.0/{f}"


In [ ]:
import gdown, os

os.makedirs("weights", exist_ok=True)

# Ré-identification (OSNet x0.25, réentraîné MSMT17) et détection football (YOLOv8m réentraîné sur
# le jeu "football-players-detection" de Roboflow) — mêmes poids que ceux validés en local, cf.
# setup.sh du dépôt.
if not os.path.isfile("weights/osnet_x0_25_msmt17.pt"):
    gdown.download("https://drive.google.com/uc?id=1Kkx2zW89jq_NETu4u42CFZTMVD5Hwm6e",
                    "weights/osnet_x0_25_msmt17.pt", quiet=False)

if not os.path.isfile("weights/yolov8m-640-football-players.pt"):
    gdown.download("https://drive.google.com/uc?id=1GWvf50u4yTep9pcF_ReDajnISsUtzvln",
                    "weights/yolov8m-640-football-players.pt", quiet=False)


## Code du pipeline

Les 5 fichiers suivants sont écrits tels quels depuis le dépôt local — même code que celui déjà
testé et commité, pas une réécriture pour Colab.

In [ ]:
%%writefile calibration.py
"""Calibration terrain image par image — wrapper autour de PnLCalib (vendored dans vendor/PnLCalib,
voir setup.sh). Une vidéo follow cam n'a pas d'homographie fixe : chaque frame est recalibrée
indépendamment à partir des lignes du terrain visibles à cet instant précis.
"""
import sys
from pathlib import Path

VENDOR = Path(__file__).parent / "vendor" / "PnLCalib"
sys.path.insert(0, str(VENDOR))

import cv2
import yaml
import torch
import numpy as np
from PIL import Image
import torchvision.transforms.functional as tvf
import torchvision.transforms as T

from model.cls_hrnet import get_cls_net
from model.cls_hrnet_l import get_cls_net as get_cls_net_l
from utils.utils_calib import FramebyFrameCalib
from utils.utils_heatmap import (
    get_keypoints_from_heatmap_batch_maxpool,
    get_keypoints_from_heatmap_batch_maxpool_l,
    complete_keypoints,
    coords_to_dict,
)

PITCH_LENGTH_M = 105.0
PITCH_WIDTH_M = 68.0

_RESIZE = T.Resize((540, 960))


class Calibrator:
    """Calibre chaque frame indépendamment (adapté à une caméra qui bouge/zoome, type follow cam)."""

    def __init__(self, frame_width, frame_height, device="cpu",
                 kp_threshold=0.3434, line_threshold=0.7867, pnl_refine=True):
        cfg = yaml.safe_load(open(VENDOR / "config" / "hrnetv2_w48.yaml"))
        cfg_l = yaml.safe_load(open(VENDOR / "config" / "hrnetv2_w48_l.yaml"))

        self.device = device
        self.kp_threshold = kp_threshold
        self.line_threshold = line_threshold
        self.pnl_refine = pnl_refine

        self.model = get_cls_net(cfg)
        self.model.load_state_dict(torch.load(VENDOR / "weights" / "SV_kp", map_location=device))
        self.model.to(device).eval()

        self.model_l = get_cls_net_l(cfg_l)
        self.model_l.load_state_dict(torch.load(VENDOR / "weights" / "SV_lines", map_location=device))
        self.model_l.to(device).eval()

        self.cam = FramebyFrameCalib(iwidth=frame_width, iheight=frame_height, denormalize=True)
        self._crash_count = 0  # cf. try/except ci-dessous — juste pour un avertissement one-shot

    def _cam_params(self, frame_bgr):
        frame = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        frame = Image.fromarray(frame)
        frame = tvf.to_tensor(frame).float().unsqueeze(0)
        frame = frame if frame.size()[-1] == 960 else _RESIZE(frame)
        frame = frame.to(self.device)
        _, _, h, w = frame.size()

        with torch.no_grad():
            heatmaps = self.model(frame)
            heatmaps_l = self.model_l(frame)

        kp_coords = get_keypoints_from_heatmap_batch_maxpool(heatmaps[:, :-1, :, :])
        line_coords = get_keypoints_from_heatmap_batch_maxpool_l(heatmaps_l[:, :-1, :, :])
        kp_dict = coords_to_dict(kp_coords, threshold=self.kp_threshold)
        lines_dict = coords_to_dict(line_coords, threshold=self.line_threshold)
        kp_dict, lines_dict = complete_keypoints(kp_dict[0], lines_dict[0], w=w, h=h, normalize=True)

        self.cam.update(kp_dict, lines_dict)
        try:
            return self.cam.heuristic_voting(refine_lines=self.pnl_refine)
        except Exception as e:
            # PnLCalib (code tiers vendored) peut planter sur un jeu de points/lignes dégénéré pour
            # une frame donnée (ex. cv2.calibrateCamera sur une frame avec très peu de repères
            # détectés) — rencontré concrètement sur un match complet (99 min), jamais sur les
            # extraits courts testés en local. Un run de plusieurs heures sans surveillance ne doit
            # pas s'arrêter pour une seule frame dégénérée : traité comme un échec de calibration
            # ordinaire (retourne None, déjà géré partout ailleurs), pas comme une erreur fatale.
            self._crash_count += 1
            if self._crash_count == 1:
                print(f"Avertissement — calibration a levé une exception sur au moins une frame "
                      f"({type(e).__name__}: {e}) ; traitée comme un échec de calibration normal, "
                      f"le run continue. Pas de message pour les occurrences suivantes.")
            return None

    def homography_pitch_to_image(self, frame_bgr):
        """3x3 : [Xc, Yc, 1] (mètres, origine au centre du terrain) -> [u, v, w] (pixels homogènes).
        None si le terrain n'est pas assez visible dans cette frame pour calibrer (fréquent en follow
        cam lors d'un plan serré sur un duel loin de toute ligne)."""
        params = self._cam_params(frame_bgr)
        if params is None:
            return None
        cp = params["cam_params"]
        rotation = np.array(cp["rotation_matrix"])
        position = np.array(cp["position_meters"])
        Q = np.array([[cp["x_focal_length"], 0, cp["principal_point"][0]],
                      [0, cp["y_focal_length"], cp["principal_point"][1]],
                      [0, 0, 1]])
        it = np.eye(4)[:-1]
        it[:, -1] = -position
        p = Q @ (rotation @ it)  # 3x4, monde -> image, origine centre terrain
        return p[:, [0, 1, 3]]  # plan Z=0 (sol) uniquement : colonnes X, Y, translation


def image_to_pitch_norm(homography_pitch_to_image, px, py):
    """Pixel (px, py) -> position terrain normalisée (0-1, convention du site : x=largeur, y=longueur
    dans le sens d'attaque, cf. BAND dans generateMockAdvancedAnalytics côté App.jsx). None si le point
    ne correspond à aucune position plausible sur le terrain (calibration probablement mauvaise)."""
    try:
        h_inv = np.linalg.inv(homography_pitch_to_image)
    except np.linalg.LinAlgError:
        return None
    world = h_inv @ np.array([px, py, 1.0])
    if abs(world[2]) < 1e-9:
        return None
    xc, yc = world[0] / world[2], world[1] / world[2]
    x_m, y_m = xc + PITCH_LENGTH_M / 2, yc + PITCH_WIDTH_M / 2
    if not (-5 <= x_m <= PITCH_LENGTH_M + 5 and -5 <= y_m <= PITCH_WIDTH_M + 5):
        return None  # projection hors terrain -> calibration probablement mauvaise pour ce point
    # PnLCalib : X = longueur (0-105, but-à-but), Y = largeur (0-68) -> site : x=largeur, y=longueur.
    # Sens d'attaque (quelle moitié = "y proche de 0") non résolu ici : simplification connue de la V1,
    # à corriger via --flip si la heatmap ressort inversée sur un match donné.
    return max(0.0, min(1.0, y_m / PITCH_WIDTH_M)), max(0.0, min(1.0, x_m / PITCH_LENGTH_M))


In [ ]:
%%writefile metrics.py
"""Accumulation des trajectoires par trace et calcul des métriques du schéma
emptyAdvancedAnalytics() (src/App.jsx:24349) — physique par joueur, heatmap, forme d'équipe.
Unités alignées sur generateMockAdvancedAnalytics() (src/App.jsx:24362) : distances en mètres,
vitesse en km/h.
"""
from collections import Counter

import numpy as np

SPRINT_SPEED_MS = 6.0          # m/s (~21.6 km/h) - seuil usuel pour compter un "sprint"
HIGH_INTENSITY_SPEED_MS = 4.5  # m/s (~16.2 km/h) - seuil "haute intensité"
ACCEL_THRESHOLD_MS2 = 2.5      # m/s^2 - seuil pour compter une accélération/décélération
SPRINT_MIN_DURATION_S = 1.0    # un pic de vitesse isolé d'un seul échantillon ne compte pas
# Vitesse humaine max plausible (record de sprint élite ~12.4 m/s) — au-delà, la variation de
# position entre deux échantillons vient presque certainement d'une calibration ponctuellement
# mauvaise (position "téléportée"), pas d'un vrai déplacement. Repéré concrètement en testant sur
# une vraie vidéo de match : sans ce filtre, une seule frame mal calibrée peut produire des
# vitesses de pointe à plusieurs centaines de km/h. Le segment est ignoré (ni distance ni vitesse),
# pas juste plafonné, pour ne pas fausser la distance totale non plus.
MAX_PLAUSIBLE_SPEED_MS = 10.5
TEAM_SHAPE_MIN_PLAYERS = 8     # nb mini de joueurs d'une même équipe visibles simultanément pour
                                # que la forme d'équipe de cette frame soit prise en compte

SMOOTH_WINDOW = 7  # nb d'échantillons (centré) pour le filtre médian anti-bruit de calibration -
                    # valeur choisie empiriquement (3/5/7/9 testés sur le même extrait réel de 90s) :
                    # meilleur résultat obtenu à 7 (couverture max 38%, vitesses redescendues à des
                    # niveaux plausibles), gains marginaux/mitigés au-delà.

PITCH_WIDTH_M = 68.0
PITCH_LENGTH_M = 105.0

# Vote majoritaire du numéro de maillot (jersey_ocr.JerseyReader) - seuils pour qu'une trace obtienne
# un numéro "confiant" utilisable comme contrainte de regroupement (tracking.cluster_tracks_globally).
# Une seule lecture, même à confiance individuelle correcte, ne suffit jamais (cf. jersey_ocr.py) :
# il faut un vrai consensus sur plusieurs lectures indépendantes.
JERSEY_MIN_CONFIDENT_READS = 3
JERSEY_MIN_READ_CONFIDENCE = 0.5
JERSEY_MIN_AGREEMENT = 0.6  # part des lectures confiantes qui doivent tomber sur le même numéro


def smooth_track_samples(samples, window=SMOOTH_WINDOW):
    """Filtre médian (par axe, fenêtre glissante centrée) sur une trace continue — corrige le
    vrai problème trouvé en testant sur un match complet : la calibration recalcule chaque frame
    indépendamment (nécessaire pour une caméra qui bouge, cf. calibration.py), donc même un joueur
    immobile peut voir sa position projetée sauter d'une frame à l'autre par simple instabilité du
    calage terrain — PAS une confusion d'identité. Vérifié concrètement : une trace jamais fusionnée
    ni découpée montrait des vitesses oscillant entre 0,9 et 42 m/s frame à frame ; un simple filtre
    médian sur fenêtre 3 fait tomber l'écrasante majorité des segments sous le seuil humain plausible
    (MAX_PLAUSIBLE_SPEED_MS), qui reste comme filet de sécurité pour ce qu'il en reste.

    Ne suppose pas un échantillonnage régulier (un index-based, pas time-based) — approximation
    acceptée : les trous (frames sans calibration) sont déjà rares individuellement dans une trace
    continue, la fenêtre glissante par index reste une bonne approximation d'une fenêtre temporelle."""
    if len(samples) < 3 or window < 3:
        return samples
    half = window // 2
    smoothed = []
    for i in range(len(samples)):
        lo, hi = max(0, i - half), min(len(samples), i + half + 1)
        window_slice = samples[lo:hi]
        xs = sorted(s[1] for s in window_slice)
        ys = sorted(s[2] for s in window_slice)
        mid = len(window_slice) // 2
        smoothed.append((samples[i][0], xs[mid], ys[mid]))
    return smoothed


class TrackAccumulator:
    """Une instance par trace (un joueur suivi/ré-identifié en flux sur tout le match — avant le
    regroupement global final, cf. tracking.cluster_tracks_globally)."""

    def __init__(self):
        self.samples = []  # [(t_seconds, x_norm, y_norm), ...] triés par t, positions calibrées
        self.team = None
        self._embedding_sum = None  # somme courante -> moyenne robuste, pas juste le dernier vu
        self._embedding_count = 0
        self.jersey_readings = []  # [(numero:str, confiance:float), ...] cf. jersey_ocr.JerseyReader

    def add_seen(self, team):
        if team and self.team is None:
            self.team = team

    def add_position(self, t, x_norm, y_norm):
        self.samples.append((t, x_norm, y_norm))

    def add_embedding(self, embedding):
        if embedding is None:
            return
        if self._embedding_sum is None:
            self._embedding_sum = embedding.copy()
        else:
            self._embedding_sum += embedding
        self._embedding_count += 1

    def add_jersey_reading(self, reading):
        if reading is not None:
            self.jersey_readings.append(reading)

    @property
    def majority_jersey(self):
        """Numéro de maillot si un consensus net se dégage sur cette trace, sinon None (pas assez
        de lectures confiantes, ou lectures trop partagées entre plusieurs numéros pour trancher) —
        cf. JERSEY_MIN_* ci-dessus. Volontairement conservateur : ce champ sert de contrainte dure
        au regroupement (deux numéros confiants différents -> jamais le même joueur), une fausse
        confiance coûterait plus cher qu'une trace laissée sans numéro."""
        confident = [(n, c) for n, c in self.jersey_readings if c >= JERSEY_MIN_READ_CONFIDENCE]
        if len(confident) < JERSEY_MIN_CONFIDENT_READS:
            return None
        counts = Counter(n for n, _ in confident)
        number, count = counts.most_common(1)[0]
        if count / len(confident) < JERSEY_MIN_AGREEMENT:
            return None
        return number

    @property
    def mean_embedding(self):
        """Embedding moyen sur toute la durée de la trace — plus robuste qu'un embedding pris sur
        une seule frame pour le regroupement global (moins sensible à un angle/une occlusion
        ponctuelle)."""
        if self._embedding_sum is None:
            return None
        mean = self._embedding_sum / self._embedding_count
        norm = np.linalg.norm(mean)
        return mean / norm if norm > 0 else None

    @property
    def time_range(self):
        if not self.samples:
            return None
        return self.samples[0][0], self.samples[-1][0]


def _speed_series_ms(samples):
    """Vitesse (m/s) entre échantillons successifs, à partir de positions normalisées 0-1
    reconverties en mètres (convention du site : x=largeur 68m, y=longueur 105m)."""
    speeds = []
    for (t0, x0, y0), (t1, x1, y1) in zip(samples, samples[1:]):
        dt = t1 - t0
        if dt <= 0:
            continue
        dx_m = (x1 - x0) * PITCH_WIDTH_M
        dy_m = (y1 - y0) * PITCH_LENGTH_M
        dist_m = (dx_m ** 2 + dy_m ** 2) ** 0.5
        v = dist_m / dt
        if v > MAX_PLAUSIBLE_SPEED_MS:
            continue  # position "téléportée" (calibration ponctuellement mauvaise) - segment ignoré
        speeds.append((t1, v, dist_m))
    return speeds  # [(t, vitesse_m_s, distance_segment_m), ...]


def compute_player_physical(samples):
    """samples : [(t_seconds, x_norm, y_norm), ...] pour une trace. Retourne le sous-objet
    'players[id]' du schéma (hors heatmapPoints/visibleCoverage, ajoutés séparément), ou None si
    la trace n'a pas assez de positions pour calculer quoi que ce soit d'utile."""
    if len(samples) < 2:
        return None
    speed_series = _speed_series_ms(samples)
    if not speed_series:
        return None

    distance_m = sum(d for _, _, d in speed_series)
    top_speed_ms = max(v for _, v, _ in speed_series)
    high_intensity_m = sum(d for _, v, d in speed_series if v >= HIGH_INTENSITY_SPEED_MS)

    sprints = 0
    in_sprint = False
    sprint_start = speed_series[0][0]
    for t, v, _ in speed_series:
        if v >= SPRINT_SPEED_MS:
            if not in_sprint:
                in_sprint = True
                sprint_start = t
        else:
            if in_sprint and (t - sprint_start) >= SPRINT_MIN_DURATION_S:
                sprints += 1
            in_sprint = False
    if in_sprint and (speed_series[-1][0] - sprint_start) >= SPRINT_MIN_DURATION_S:
        sprints += 1

    accelerations = decelerations = 0
    for (t0, v0, _), (t1, v1, _) in zip(speed_series, speed_series[1:]):
        dt = t1 - t0
        if dt <= 0:
            continue
        a = (v1 - v0) / dt
        if a >= ACCEL_THRESHOLD_MS2:
            accelerations += 1
        elif a <= -ACCEL_THRESHOLD_MS2:
            decelerations += 1

    return {
        "distanceCovered": round(distance_m),
        "sprints": sprints,
        "topSpeed": round(top_speed_ms * 3.6, 1),
        "highIntensityDistance": round(high_intensity_m),
        "accelerations": accelerations,
        "decelerations": decelerations,
    }


def _drop_implausible_points(samples):
    """Écarte un point si le déplacement vers OU depuis un voisin immédiat dépasse la vitesse
    humaine plausible — le même filtre que _speed_series_ms applique déjà aux stats agrégées
    (distance, vitesse), mais qui ne touchait jusqu'ici pas la heatmap : un point "téléporté"
    pouvait rester affiché même une fois exclu du calcul de distance/vitesse. Repéré concrètement
    sur le premier match complet traité (des sauts à 27-40 m/s dans les points bruts d'une trace
    par ailleurs déjà fusionnée par le regroupement global — signe probable d'une fusion encore
    incorrecte de deux joueurs différents, cf. discussion avec Gregory)."""
    if len(samples) < 2:
        return samples
    n = len(samples)
    keep = [True] * n
    for i in range(n - 1):
        t0, x0, y0 = samples[i]
        t1, x1, y1 = samples[i + 1]
        dt = t1 - t0
        if dt <= 0:
            continue
        dist_m = (((x1 - x0) * PITCH_WIDTH_M) ** 2 + ((y1 - y0) * PITCH_LENGTH_M) ** 2) ** 0.5
        if dist_m / dt > MAX_PLAUSIBLE_SPEED_MS:
            keep[i] = keep[i + 1] = False
    return [s for s, k in zip(samples, keep) if k]


def compute_heatmap_points(samples, max_points=200):
    """Sous-échantillonne si besoin — le site n'a pas besoin de milliers de points pour une
    heatmap lisible (generateMockAdvancedAnalytics() en génère ~20-30 par joueur)."""
    samples = _drop_implausible_points(samples)
    if not samples:
        return []
    step = max(1, len(samples) // max_points)
    return [{"x": round(x, 3), "y": round(y, 3), "t": round(t)} for t, x, y in samples[::step]]


def compute_team_shape(frame_team_positions):
    """frame_team_positions : une entrée par frame échantillonnée où AU MOINS UN joueur de cette
    équipe a été positionné, chaque entrée étant la liste de ses (x_norm, y_norm) ce coup-ci. Les
    frames avec trop peu de joueurs visibles ensemble sont ignorées (cf. TEAM_SHAPE_MIN_PLAYERS) —
    sur une vidéo follow cam, la majorité des frames n'ont pas toute l'équipe à l'écran, donc cette
    métrique est échantillonnée sur les moments où le plan est assez large, pas continue."""
    heights, widths, depths = [], [], []
    for positions in frame_team_positions:
        if len(positions) < TEAM_SHAPE_MIN_PLAYERS:
            continue
        xs = [x for x, _ in positions]
        ys = [y for _, y in positions]
        heights.append(sum(ys) / len(ys))
        widths.append(max(xs) - min(xs))
        depths.append(max(ys) - min(ys))
    if not heights:
        return {"avgBlockHeight": None, "avgWidth": None, "avgDepth": None, "ppda": None}
    return {
        "avgBlockHeight": round(sum(heights) / len(heights), 3),
        "avgWidth": round(sum(widths) / len(widths), 3),
        "avgDepth": round(sum(depths) / len(depths), 3),
        "ppda": None,  # nécessite une détection d'actions défensives - hors scope de cette V1
    }


In [ ]:
%%writefile tracking.py
"""Détection (YOLO), tracking (ByteTrack via le package `trackers`) et affectation d'équipe
(couleur de maillot) par frame.

Note API : `supervision.ByteTrack` est déprécié depuis la 0.28 (retrait en 0.31) au profit du
package `trackers` (`ByteTrackTracker`, méthode `update()` au lieu de `update_with_detections()`)
— vérifié directement dans le code installé plutôt que supposé, l'un ne redirige pas vers l'autre.

Ré-identification (ReIdentifier) : par embedding d'apparence (torchreid/OSNet), pas par couleur
brute. Validé empiriquement sur un vrai extrait (ALDM-FCSN) avant intégration : similarité cosinus
moyenne ~0.88 entre crops d'un même joueur, ~0.59 entre joueurs différents — nette séparation. La
couleur de maillot reste utilisée séparément pour l'affectation d'équipe (TeamAssigner), qui est un
problème plus simple (2 classes déséquilibrées par les couleurs, pas une identité individuelle).
"""
from pathlib import Path

import cv2
import numpy as np
import torch
import supervision as sv
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
from ultralytics import YOLO
from trackers import ByteTrackTracker
from torchreid.reid.utils import FeatureExtractor

from metrics import TrackAccumulator, MAX_PLAUSIBLE_SPEED_MS, PITCH_WIDTH_M, PITCH_LENGTH_M

DEFAULT_DETECTION_WEIGHTS = Path(__file__).parent / "weights" / "yolov8m-640-football-players.pt"
# Classes du modèle football dédié (Darkmyter/Football-Players-Tracking, YOLOv8m réentraîné sur le
# jeu "football-players-detection" de Roboflow) — vérifiées directement via model.names, PAS l'ordre
# du README. Détecte nettement plus de joueurs que YOLO générique + classe COCO "person" (jusqu'à
# +60-70% sur des frames réelles testées) et sépare nativement arbitre/gardien/joueur/ballon, ce qui
# permet d'exclure les arbitres du suivi plutôt que de les compter comme des joueurs.
BALL_CLASS = 0
PLAYER_CLASSES = [1, 2]  # goalkeeper, player — les deux comptent pour les stats physiques
REFEREE_CLASS = 3        # explicitement exclu du tracking

REID_WEIGHTS = Path(__file__).parent / "weights" / "osnet_x0_25_msmt17.pt"
REID_MAX_GAP_SECONDS = 90.0    # signal bien plus fiable que la couleur -> fenêtre élargie
REID_EMBEDDING_MIN_SIM = 0.75  # similarité cosinus mini pour relier deux traces (cf. validation ci-dessus)
# Regroupement global (cluster_tracks_globally) : seuil plus strict que REID_EMBEDDING_MIN_SIM.
# La validation initiale (0.88 même trace / 0.59 traces différentes) mélangeait des paires de
# n'importe quelle équipe — beaucoup faciles à distinguer par la seule couleur de maillot. Le
# regroupement global ne compare QUE des joueurs de la MÊME équipe (le maillot ne différencie donc
# plus rien), une tâche plus dure où deux joueurs différents peuvent être plus proches que prévu.
# Repéré concrètement : avec 0.75 ici, plusieurs joueurs réels différents ont été fusionnés en une
# seule trace (couverture résultante > 1.0, mathématiquement impossible — bug corrigé).
REID_GLOBAL_MIN_SIM = 0.88


def _shirt_color(frame_bgr, box):
    """Couleur moyenne (HSV) du tiers supérieur de la boîte — approxime le maillot plutôt que le short.
    Sert uniquement à l'affectation d'équipe (TeamAssigner), pas à la ré-identification individuelle."""
    x1, y1, x2, y2 = [int(v) for v in box]
    y2_shirt = y1 + max(1, (y2 - y1) // 3)
    x1, y1 = max(0, x1), max(0, y1)
    x2, y2_shirt = min(frame_bgr.shape[1], x2), min(frame_bgr.shape[0], y2_shirt)
    if x2 <= x1 or y2_shirt <= y1:
        return None
    crop = frame_bgr[y1:y2_shirt, x1:x2]
    hsv = cv2.cvtColor(crop, cv2.COLOR_BGR2HSV)
    return hsv.reshape(-1, 3).mean(axis=0)


def _crop(frame_bgr, box):
    x1, y1, x2, y2 = [int(v) for v in box]
    x1, y1 = max(0, x1), max(0, y1)
    x2, y2 = min(frame_bgr.shape[1], x2), min(frame_bgr.shape[0], y2)
    if x2 <= x1 or y2 <= y1:
        return None
    return frame_bgr[y1:y2, x1:x2]


class TeamAssigner:
    """Calibre les deux couleurs d'équipe sur les premières détections rencontrées, puis affecte
    chaque trace à l'équipe la plus proche ('autre' si trop loin des deux — arbitre probable)."""

    def __init__(self, calibration_samples=60):
        self.calibration_samples = calibration_samples
        self._samples = []
        self.centers = None  # (2, 3) HSV

    def observe(self, color):
        if color is not None and self.centers is None:
            self._samples.append(color)
            if len(self._samples) >= self.calibration_samples:
                self._fit()

    def _fit(self):
        data = np.array(self._samples, dtype=np.float32)
        criteria = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 50, 0.5)
        _, _, centers = cv2.kmeans(data, 2, None, criteria, 5, cv2.KMEANS_PP_CENTERS)
        self.centers = centers

    def assign(self, color):
        if self.centers is None or color is None:
            return None
        d0 = float(np.linalg.norm(color - self.centers[0]))
        d1 = float(np.linalg.norm(color - self.centers[1]))
        if min(d0, d1) > 80:  # trop loin des deux équipes -> probablement un arbitre
            return "autre"
        return "A" if d0 < d1 else "B"


class ReIdentifier:
    """Ré-identification par embedding d'apparence (OSNet) : relie une trace qui réapparaît après
    être sortie du cadre à son identité précédente plutôt que de lui en attribuer une nouvelle.
    Reste imparfait (deux joueurs très similaires d'apparence peuvent se confondre), mais nettement
    plus fiable qu'un simple appariement de couleur moyenne — cf. validation en tête de fichier."""

    def __init__(self):
        self.lost = {}    # canonical_id -> {"embedding", "team", "t"}
        self.remap = {}   # id ByteTrack -> canonical_id déjà résolu
        # Diagnostic : combien de fois resolve() a effectivement retrouvé un candidat, vs combien de
        # fois une trace ByteTrack jamais vue avait au moins un candidat récent de la même équipe
        # disponible (donc une vraie occasion de fusionner, ratée ou réussie) — permet de savoir si
        # la ré-identification est le facteur limitant ou si le décrochage se produit ailleurs
        # (perte interne à ByteTrack, trop rapide pour même atteindre cette logique).
        self.merges = 0
        self.opportunities = 0

    def canonical(self, track_id):
        return self.remap.get(track_id, track_id)

    def mark_lost(self, track_id, embedding, team, t):
        if embedding is None:
            return
        self.lost[self.canonical(track_id)] = {"embedding": embedding, "team": team, "t": t}

    def resolve(self, track_id, embedding, team, t):
        """Trace ByteTrack jamais vue jusqu'ici : cherche un candidat perdu récemment, même équipe,
        embedding proche. Sinon la trace reste nouvelle (nouveau joueur, ou ré-identification ratée)."""
        if track_id in self.remap:
            return self.remap[track_id]
        candidate_seen = False
        best_cid, best_sim = None, REID_EMBEDDING_MIN_SIM
        for cid, info in list(self.lost.items()):
            if t - info["t"] > REID_MAX_GAP_SECONDS:
                del self.lost[cid]
                continue
            if embedding is None or info["team"] != team:
                continue
            candidate_seen = True
            sim = float(np.dot(embedding, info["embedding"]))  # vecteurs déjà normalisés
            if sim > best_sim:
                best_sim, best_cid = sim, cid
        if candidate_seen:
            self.opportunities += 1
        if best_cid is not None:
            self.merges += 1
            self.remap[track_id] = best_cid
            del self.lost[best_cid]
            return best_cid
        return track_id


class Tracker:
    def __init__(self, model_path=None, device="cpu", confidence=0.3, frame_rate=25.0):
        model_path = model_path or str(DEFAULT_DETECTION_WEIGHTS)
        if model_path == str(DEFAULT_DETECTION_WEIGHTS) and not DEFAULT_DETECTION_WEIGHTS.exists():
            raise FileNotFoundError(
                f"Poids de détection football manquants : {DEFAULT_DETECTION_WEIGHTS} — lance setup.sh."
            )
        self.model = YOLO(model_path)
        self.device = device
        self.confidence = confidence
        self.byte_track = ByteTrackTracker(frame_rate=frame_rate, lost_track_buffer=int(frame_rate * 3))
        self.team_assigner = TeamAssigner()
        self.reid = ReIdentifier()
        self._last_seen = {}  # track_id ByteTrack -> (embedding, team), pour marquer "lost" au bon moment
        self._active_last_frame = set()
        # Équipe verrouillée par identité canonique dès la 1re classification réussie : un maillot ne
        # change pas de couleur en cours de match, donc on ne veut pas qu'un même joueur soit reclassé
        # différemment (et donc fragmenté dans les accumulateurs) si une frame donnée classe mal sa
        # couleur — repéré concrètement en testant sur un match réel noir/blanc, où la teinte HSV est
        # peu fiable pour distinguer deux couleurs proches de l'achromatique.
        self._known_team = {}
        if not REID_WEIGHTS.exists():
            raise FileNotFoundError(
                f"Poids de ré-identification manquants : {REID_WEIGHTS} — lance setup.sh."
            )
        self.reid_extractor = FeatureExtractor(
            model_name="osnet_x0_25", model_path=str(REID_WEIGHTS), device=device, verbose=False
        )

    def _embeddings(self, frame_bgr, boxes, indices):
        """Extrait un embedding normalisé par boîte (indices = positions dans `boxes` à traiter),
        en un seul appel batché — nettement plus rapide que boîte par boîte."""
        crops, valid = [], []
        for i in indices:
            c = _crop(frame_bgr, boxes[i])
            if c is not None:
                crops.append(cv2.cvtColor(c, cv2.COLOR_BGR2RGB))
                valid.append(i)
        if not crops:
            return {}
        with torch.no_grad():
            feats = self.reid_extractor(crops)
            feats = feats / feats.norm(dim=1, keepdim=True)
        return {i: feats[k].cpu().numpy() for k, i in enumerate(valid)}

    def process_frame(self, frame_bgr, t_seconds):
        """Retourne (joueurs, position_ballon).
        joueurs = liste de {"track_id": id stable ré-identifié, "team": "A"/"B"/"autre"/None,
                             "px": float, "py": float, "box": (x1,y1,x2,y2), "embedding": array ou None}
                             (px, py = pieds au sol, en pixels image ; box = boîte détectée brute)
        position_ballon = (px, py) en pixels, ou None si aucun ballon détecté cette frame."""
        # Arbitre volontairement absent de `classes` : jamais suivi comme joueur.
        result = self.model(frame_bgr, device=self.device, verbose=False,
                             classes=PLAYER_CLASSES + [BALL_CLASS])[0]
        detections = sv.Detections.from_ultralytics(result)
        detections = detections[detections.confidence > self.confidence]

        people = detections[np.isin(detections.class_id, PLAYER_CLASSES)]
        balls = detections[detections.class_id == BALL_CLASS]
        # `frame=` n'est pas utilisé par l'estimateur d'état par défaut (avertissement sinon) —
        # notre ReIdentifier gère la ré-identification par apparence séparément.
        tracked = self.byte_track.update(people, timestamp=t_seconds)

        confirmed = [i for i, tid in enumerate(tracked.tracker_id) if tid is not None and tid >= 0]
        embeddings = self._embeddings(frame_bgr, tracked.xyxy, confirmed)

        active_now = set()
        out_players = []
        for i in confirmed:
            box, track_id = tracked.xyxy[i], int(tracked.tracker_id[i])
            embedding = embeddings.get(i)
            color = _shirt_color(frame_bgr, box)
            self.team_assigner.observe(color)
            team = self.team_assigner.assign(color)

            canonical_id = self.reid.resolve(track_id, embedding, team, t_seconds)
            if canonical_id in self._known_team:
                team = self._known_team[canonical_id]
            elif team is not None:
                self._known_team[canonical_id] = team
            active_now.add(track_id)
            self._last_seen[track_id] = (embedding, team)

            out_players.append({
                "track_id": canonical_id,
                "team": team,
                "px": float((box[0] + box[2]) / 2),
                "py": float(box[3]),
                "box": tuple(float(v) for v in box),
                "embedding": embedding,  # pour le regroupement global final (cf. cluster_tracks_globally)
            })

        for lost_id in self._active_last_frame - active_now:
            embedding, team = self._last_seen.get(lost_id, (None, None))
            self.reid.mark_lost(lost_id, embedding, team, t_seconds)
        self._active_last_frame = active_now

        ball_xy = None
        if len(balls) > 0:
            best = int(np.argmax(balls.confidence))
            bx = balls.xyxy[best]
            ball_xy = (float((bx[0] + bx[2]) / 2), float((bx[1] + bx[3]) / 2))

        return out_players, ball_xy


def _boundary_transition_plausible(acc_earlier, acc_later):
    """Le passage direct entre le dernier point de la trace la plus ancienne et le premier point
    de l'autre doit rester physiquement possible pour un humain (même limite que celle qui filtre
    déjà les segments à l'intérieur d'une trace, cf. metrics.MAX_PLAUSIBLE_SPEED_MS) — sinon les
    deux traces ne peuvent pas être le même joueur, quelle que soit la ressemblance d'apparence.

    Repéré concrètement sur le premier match complet traité : une trace déjà passée par ce
    regroupement affichait des sauts à 27-40 m/s entre deux points, largement au-dessus du possible
    humain (record du monde ~10,4 m/s) — la seule similarité d'embedding ne suffit pas, deux joueurs
    différents de la même équipe peuvent se ressembler assez pour dépasser le seuil."""
    t0, x0, y0 = acc_earlier.samples[-1]
    t1, x1, y1 = acc_later.samples[0]
    dt = t1 - t0
    if dt <= 0:
        return False
    dist_m = (((x1 - x0) * PITCH_WIDTH_M) ** 2 + ((y1 - y0) * PITCH_LENGTH_M) ** 2) ** 0.5
    return dist_m / dt <= MAX_PLAUSIBLE_SPEED_MS


def split_implausible_tracks(accumulators):
    """À appeler AVANT cluster_tracks_globally() : coupe une trace en plusieurs morceaux partout où
    un déplacement interne dépasse la vitesse humaine plausible. Une telle rupture signifie presque
    certainement que la ré-identification EN FLUX (ReIdentifier, dans Tracker.process_frame) a déjà
    relié à tort deux joueurs réels différents avant même d'atteindre le regroupement global — cette
    étape-là ne peut décider que de fusionner ou non des traces déjà propres, pas réparer une trace
    déjà contaminée à la source. Repéré concrètement sur le premier match complet traité : des sauts
    de 27-40 m/s subsistaient dans des traces jamais passées par le regroupement global (donc jamais
    vues par _boundary_transition_plausible), preuve que la contamination venait d'en amont.

    Chaque morceau hérite du même embedding moyen que la trace d'origine — conserver un embedding
    précis par morceau demanderait de garder celui de chaque frame individuellement plutôt qu'une
    simple somme courante. Approximation acceptée : cluster_tracks_globally() revalidera de toute
    façon chaque paire de morceaux avec la même contrainte physique avant de les refusionner."""
    result = {}
    for key, acc in accumulators.items():
        if len(acc.samples) < 2:
            result[key] = acc
            continue

        segments, current = [], [acc.samples[0]]
        for prev, cur in zip(acc.samples, acc.samples[1:]):
            dt = cur[0] - prev[0]
            dist_m = (((cur[1] - prev[1]) * PITCH_WIDTH_M) ** 2
                      + ((cur[2] - prev[2]) * PITCH_LENGTH_M) ** 2) ** 0.5
            if dt > 0 and dist_m / dt > MAX_PLAUSIBLE_SPEED_MS:
                segments.append(current)
                current = []
            current.append(cur)
        segments.append(current)

        if len(segments) == 1:
            result[key] = acc
            continue
        for i, seg in enumerate(segments):
            piece = TrackAccumulator()
            piece.team = acc.team
            piece.samples = seg
            piece._embedding_sum = acc._embedding_sum
            piece._embedding_count = acc._embedding_count
            piece.jersey_readings = acc.jersey_readings
            result[f"{key}~{i}"] = piece
    return result


def cluster_tracks_globally(accumulators, min_sim=REID_GLOBAL_MIN_SIM):
    """Regroupe après coup les traces qui appartiennent probablement au même joueur réel, en
    comparant TOUTES les paires de traces du match (pas seulement celles proches dans le temps,
    contrairement à ReIdentifier en flux). Nécessaire car la probabilité qu'AU MOINS UNE
    ré-identification en flux échoue augmente avec le nombre de fois qu'un joueur sort du cadre —
    sur un match complet, même un taux de réussite correct par tentative (~50-60%, mesuré) laisse
    presque certainement passer au moins un échec (0.6^5 ≈ 8% de réussir 5 fois d'affilée). Un
    passage global ne dépend plus de ce cumul : une seule comparaison finale par paire suffit,
    indépendamment du nombre de ruptures survenues en flux.

    Clustering à liaison complète (scipy), pas un simple union-find : un union-find fusionne dès
    qu'une CHAÎNE de paires similaires existe (A~B, B~C -> A,C regroupés même si A et C sont très
    différents), ce qui a produit en pratique des couvertures > 1.0 (plusieurs joueurs réels
    fusionnés en une trace, détecté et corrigé). La liaison complète exige que TOUS les membres
    d'un groupe restent proches les uns des autres, pas juste chaînés.

    Deux véto absolus avant même de regarder la similarité d'apparence : chevauchement temporel
    (impossible d'être la même personne à deux endroits en même temps), et transition physiquement
    impossible entre la fin d'une trace et le début de l'autre (cf. _boundary_transition_plausible)
    — repéré sur un vrai match complet : la seule similarité d'embedding, même à un seuil strict,
    laisse passer des fusions entre deux joueurs différents mais qui se ressemblent.

    accumulators : dict label -> TrackAccumulator (label du type "A#18" — le préfixe avant "#" sert
    de clé d'équipe, on ne regroupe jamais entre équipes différentes ; le maillot ne différencie
    plus rien à l'intérieur d'une équipe, d'où un seuil plus strict que la ré-id en flux, cf.
    REID_GLOBAL_MIN_SIM). Retourne un nouveau dict, même format, traces fusionnées quand pertinent."""
    clusterable = [k for k, acc in accumulators.items()
                   if acc.mean_embedding is not None and acc.time_range is not None]

    by_team = {}
    for k in clusterable:
        by_team.setdefault(k.split("#", 1)[0], []).append(k)

    merged = {}
    for keys in by_team.values():
        if len(keys) == 1:
            merged[keys[0]] = accumulators[keys[0]]
            continue

        embeddings = np.stack([accumulators[k].mean_embedding for k in keys])
        dist = 1.0 - embeddings @ embeddings.T  # cosine -> distance, embeddings déjà normalisés
        # Deux morceaux issus d'un même split_implausible_tracks() héritent du même embedding
        # exact (cf. ce module) : leur similarité peut ressortir infinitésimalement au-dessus de 1.0
        # par imprécision flottante, ce qui produirait une distance négative — scipy refuse
        # catégoriquement (ValueError) plutôt que de tolérer ce bruit numérique habituel.
        dist = np.clip(dist, 0.0, None)
        ranges = [accumulators[k].time_range for k in keys]
        for i in range(len(keys)):
            for j in range(i + 1, len(keys)):
                (t1_min, t1_max), (t2_min, t2_max) = ranges[i], ranges[j]
                if t1_min <= t2_max and t2_min <= t1_max:
                    dist[i, j] = dist[j, i] = 10.0  # chevauchement temporel -> jamais le même joueur
                    continue
                acc_i, acc_j = accumulators[keys[i]], accumulators[keys[j]]
                earlier, later = (acc_i, acc_j) if t1_max <= t2_min else (acc_j, acc_i)
                if not _boundary_transition_plausible(earlier, later):
                    dist[i, j] = dist[j, i] = 10.0  # transition physiquement impossible -> jamais fusionner

        # Numéro de maillot (vote majoritaire, cf. TrackAccumulator.majority_jersey) : contrainte
        # complémentaire à l'apparence, forte précisément là où l'apparence est aveugle (deux
        # coéquipiers en maillot identique mais numéros différents, cf. jersey_ocr.py). Deux numéros
        # confiants différents -> jamais le même joueur, quelle que soit la similarité d'embedding.
        # Deux numéros confiants identiques -> fusion forcée (dist=0) SAUF si déjà vétoée ci-dessus
        # (temps/physique restent prioritaires : un OCR d'accord des deux côtés ne rend pas une
        # téléportation possible).
        for i in range(len(keys)):
            for j in range(i + 1, len(keys)):
                if dist[i, j] >= 10.0:
                    continue
                num_i = accumulators[keys[i]].majority_jersey
                num_j = accumulators[keys[j]].majority_jersey
                if num_i is None or num_j is None:
                    continue
                dist[i, j] = dist[j, i] = 0.0 if num_i == num_j else 10.0
        np.fill_diagonal(dist, 0.0)

        condensed = squareform(dist, checks=False)
        tree = linkage(condensed, method="complete")
        labels = fcluster(tree, t=1.0 - min_sim, criterion="distance")

        groups = {}
        for key, label in zip(keys, labels):
            groups.setdefault(label, []).append(key)
        for members in groups.values():
            if len(members) == 1:
                merged[members[0]] = accumulators[members[0]]
                continue
            combined = TrackAccumulator()
            combined.team = accumulators[members[0]].team
            for m in members:
                combined.samples.extend(accumulators[m].samples)
            combined.samples.sort(key=lambda s: s[0])
            merged[members[0]] = combined

    # Traces sans embedding exploitable (rare : tous les recadrages de cette trace ont échoué) —
    # laissées telles quelles, non regroupables faute de signal.
    for k, acc in accumulators.items():
        if k not in clusterable:
            merged[k] = acc

    return merged


In [ ]:
%%writefile overlay.py
"""Rendu d'un aperçu annoté (joueurs suivis + mini-terrain calibré) pour valider visuellement
détection/tracking/calibration sur un extrait court avant de lancer un match complet — cf. étape
de vérification du plan phase 1."""
import av
import cv2

TEAM_COLORS = {"A": (60, 180, 255), "B": (255, 120, 60), "autre": (200, 200, 200), None: (140, 140, 140)}

PITCH_INSET_W, PITCH_INSET_H = 220, 150
PITCH_MARGIN = 14


def _draw_pitch_inset(frame, positions_by_team):
    h, w = frame.shape[:2]
    x0, y0 = w - PITCH_INSET_W - PITCH_MARGIN, h - PITCH_INSET_H - PITCH_MARGIN
    overlay = frame.copy()
    cv2.rectangle(overlay, (x0, y0), (x0 + PITCH_INSET_W, y0 + PITCH_INSET_H), (30, 110, 30), -1)
    cv2.addWeighted(overlay, 0.75, frame, 0.25, 0, dst=frame)
    cv2.rectangle(frame, (x0, y0), (x0 + PITCH_INSET_W, y0 + PITCH_INSET_H), (255, 255, 255), 1)
    cv2.line(frame, (x0 + PITCH_INSET_W // 2, y0), (x0 + PITCH_INSET_W // 2, y0 + PITCH_INSET_H),
             (255, 255, 255), 1)
    for team, positions in positions_by_team.items():
        color = TEAM_COLORS.get(team, (200, 200, 200))
        for x_norm, y_norm in positions:
            px, py = int(x0 + x_norm * PITCH_INSET_W), int(y0 + y_norm * PITCH_INSET_H)
            cv2.circle(frame, (px, py), 3, color, -1)


def draw_debug_frame(frame, players, homography_found, positions_by_team):
    """players : sortie de Tracker.process_frame (px, py = pieds au sol, en pixels image).
    positions_by_team : {"A"/"B": [(x_norm, y_norm), ...]} pour ce même instant, déjà calibrées.
    Retourne une NOUVELLE frame annotée (ne modifie pas l'originale)."""
    frame = frame.copy()
    for p in players:
        color = TEAM_COLORS.get(p["team"], (200, 200, 200))
        x, y = int(p["px"]), int(p["py"])
        cv2.circle(frame, (x, y), 4, color, -1)
        cv2.putText(frame, str(p["track_id"]), (x + 6, y - 6), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

    status = "calibration OK" if homography_found else "calibration ECHEC (frame ignoree)"
    status_color = (80, 220, 80) if homography_found else (60, 60, 230)
    cv2.putText(frame, status, (14, 28), cv2.FONT_HERSHEY_SIMPLEX, 0.7, status_color, 2)

    _draw_pitch_inset(frame, positions_by_team)
    return frame


class DebugVideoWriter:
    """Remplace cv2.VideoWriter (fourcc 'mp4v') : ce codec produit un .mp4 valide mais illisible par
    QuickTime/Photos sur macOS (écran vert constaté sur le run complet) — H.264/yuv420p via PyAV
    (déjà présent, dépendance transitive de `supervision`) est lisible nativement partout."""

    def __init__(self, path, fps, width, height):
        self.container = av.open(path, mode="w")
        self.stream = self.container.add_stream("libx264", rate=max(1, round(fps)))
        self.stream.width = width
        self.stream.height = height
        self.stream.pix_fmt = "yuv420p"
        self.stream.options = {"crf": "23", "preset": "fast"}

    def write(self, frame_bgr):
        frame = av.VideoFrame.from_ndarray(frame_bgr, format="bgr24")
        for packet in self.stream.encode(frame):
            self.container.mux(packet)

    def release(self):
        for packet in self.stream.encode():
            self.container.mux(packet)
        self.container.close()


In [ ]:
%%writefile extract.py
#!/usr/bin/env python3
"""Pipeline d'extraction — vidéo de match -> JSON conforme à emptyAdvancedAnalytics() (src/App.jsx).

Portée phase 1 (cf. plan) : physique par joueur, heatmap, forme d'équipe uniquement — pas
d'événements, pas de xG, pas de réseau de passes, pas de PPDA. Conçu pour une caméra qui bouge/
zoome (follow cam), pas un plan large fixe : calibration terrain refaite à chaque frame, joueurs
hors champ une partie du match (cf. visibleCoverage par joueur en sortie).

Usage :
  python extract.py --video match.mp4 --out resultat.json
  python extract.py --video extrait.mp4 --out test.json --max-seconds 180 --device mps

Sans --roster : identités anonymes ("A#3", "B#11", ...) — étape 1a du plan (valider détection +
tracking + calibration avant d'investir dans le rattachement aux vrais joueurs).
Avec --roster mapping.json : fichier {"A": {"<numero>": "<playerId>"}, "B": {...}} — le camp "A"
se construit directement à partir de match.playerAssignments (Studio -> Composition, tel que déjà
saisi dans le site) — étape 1b, sortie indexée par player.id réel.
"""
import argparse
import json
import os
import pickle
from collections import defaultdict

import cv2
from tqdm import tqdm

from calibration import Calibrator, image_to_pitch_norm
from tracking import Tracker, cluster_tracks_globally, split_implausible_tracks
from metrics import (
    TrackAccumulator, compute_player_physical, compute_heatmap_points, compute_team_shape,
    smooth_track_samples,
)
from overlay import draw_debug_frame, DebugVideoWriter
from jersey_ocr import JerseyReader

# Taux de détection réel mesuré sur extrait (seuils EasyOCR assouplis, cf. jersey_ocr.py) : ~1-2%
# des tentatives seulement aboutissent à une lecture exploitable (numéro rarement visible/net -
# joueur de dos, en mouvement, loin caméra) - même les traces les plus longues du match n'auraient
# pas assez de tentatives pour atteindre un vote majoritaire (JERSEY_MIN_CONFIDENT_READS) si on
# sous-échantillonne encore l'OCR en plus de l'échantillonnage vidéo déjà réduit. D'où 1 (aucun
# sous-échantillonnage) plutôt qu'un débit réduit comme pour d'autres signaux - le surcoût mesuré
# reste faible (l'OCR ne domine pas le temps de calcul face à détection/tracking/calibration).
JERSEY_OCR_EVERY_N = 1


def _label(team, track_id):
    return f"{team or '?'}#{track_id}"


def _save_checkpoint(path, state):
    """Écriture atomique (fichier temporaire puis renommage) pour ne jamais laisser un checkpoint
    à moitié écrit si le process s'arrête pendant la sauvegarde — un run long (match complet sur
    Colab) peut se faire couper à tout moment (déconnexion de session, mise en veille de la machine
    qui héberge le navigateur)."""
    tmp = f"{path}.tmp"
    with open(tmp, "wb") as fh:
        pickle.dump(state, fh)
    os.replace(tmp, path)


def _load_checkpoint(path):
    with open(path, "rb") as fh:
        return pickle.load(fh)


def run(video_path, sample_fps, device, max_seconds=None, debug_overlay_path=None, start_seconds=0.0,
        checkpoint_path=None, checkpoint_every=300, resume=False):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise SystemExit(f"Impossible d'ouvrir la vidéo : {video_path}")
    native_fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frame_step = max(1, round(native_fps / sample_fps))

    # Reprise après coupure (déconnexion Colab, mise en veille de la machine qui héberge le
    # navigateur — vécu deux fois en local pendant ce chantier) : un run long doit pouvoir repartir
    # d'un checkpoint plutôt que de tout refaire depuis le début.
    accumulators = {}
    team_frame_positions = defaultdict(list)
    total_sampled = 0
    calibrated_sampled = 0
    reid_merges = reid_opportunities = 0
    resume_frame_idx = None
    if resume and checkpoint_path and os.path.isfile(checkpoint_path):
        state = _load_checkpoint(checkpoint_path)
        accumulators = state["accumulators"]
        team_frame_positions = defaultdict(list, state["team_frame_positions"])
        total_sampled = state["total_sampled"]
        calibrated_sampled = state["calibrated_sampled"]
        reid_merges = state["reid_merges"]
        reid_opportunities = state["reid_opportunities"]
        resume_frame_idx = state["frame_idx"]
        print(f"Reprise depuis le checkpoint : {total_sampled} frames déjà traitées, "
              f"{len(accumulators)} traces en cours.")

    if resume_frame_idx is not None:
        cap.set(cv2.CAP_PROP_POS_FRAMES, resume_frame_idx)
    elif start_seconds:
        cap.set(cv2.CAP_PROP_POS_MSEC, start_seconds * 1000)
    frame_idx = int(cap.get(cv2.CAP_PROP_POS_FRAMES))  # position réelle après seek (keyframe le plus proche)
    max_frames = frame_idx + int(max_seconds * native_fps) if max_seconds else total_frames

    calibrator = Calibrator(width, height, device=device)
    # frame_rate décrit la base de temps des `timestamp=` passés à chaque update() (le fps natif de
    # la vidéo), pas la fréquence à laquelle on appelle update() (fréquence d'échantillonnage,
    # généralement plus basse) — les deux sont indépendants une fois qu'on fournit un timestamp
    # explicite. Les confondre fait échouer la confirmation de toute trace (vérifié empiriquement).
    tracker = Tracker(device=device, frame_rate=native_fps)
    jersey_reader = JerseyReader(device=device)
    # Repartir d'un checkpoint perd l'état interne du tracker (couleurs d'équipe calibrées, galerie
    # de traces "perdues" récemment) — recalibré en quelques secondes, effet secondaire mineur
    # accepté plutôt que de sérialiser des modèles PyTorch entiers à chaque checkpoint.
    tracker.reid.merges, tracker.reid.opportunities = reid_merges, reid_opportunities

    pbar = tqdm(total=min(total_frames, max_frames - frame_idx) // frame_step, desc="Extraction")

    overlay_writer = None
    if debug_overlay_path:
        overlay_writer = DebugVideoWriter(debug_overlay_path, sample_fps, width, height)

    while True:
        ret, frame = cap.read()
        if not ret or frame_idx >= max_frames:
            break
        if frame_idx % frame_step != 0:
            frame_idx += 1
            continue

        t = frame_idx / native_fps
        total_sampled += 1

        players, _ball_xy = tracker.process_frame(frame, t)
        homography = calibrator.homography_pitch_to_image(frame)
        if homography is not None:
            calibrated_sampled += 1

        frame_positions_by_team = defaultdict(list)
        for p in players:
            key = _label(p["team"], p["track_id"])
            acc = accumulators.setdefault(key, TrackAccumulator())
            acc.add_seen(p["team"])
            acc.add_embedding(p["embedding"])
            if total_sampled % JERSEY_OCR_EVERY_N == 0:
                acc.add_jersey_reading(jersey_reader.read(frame, p["box"]))
            if homography is not None:
                pos = image_to_pitch_norm(homography, p["px"], p["py"])
                if pos is not None:
                    acc.add_position(t, *pos)
                    if p["team"] in ("A", "B"):
                        frame_positions_by_team[p["team"]].append(pos)
        for team, positions in frame_positions_by_team.items():
            team_frame_positions[team].append(positions)

        if checkpoint_path and total_sampled % checkpoint_every == 0:
            _save_checkpoint(checkpoint_path, {
                "accumulators": accumulators,
                "team_frame_positions": dict(team_frame_positions),
                "total_sampled": total_sampled,
                "calibrated_sampled": calibrated_sampled,
                "reid_merges": tracker.reid.merges,
                "reid_opportunities": tracker.reid.opportunities,
                "frame_idx": frame_idx + 1,
            })

        if overlay_writer is not None:
            overlay_writer.write(draw_debug_frame(frame, players, homography is not None, frame_positions_by_team))

        frame_idx += 1
        pbar.update(1)

    pbar.close()
    cap.release()
    if overlay_writer is not None:
        overlay_writer.release()
    return accumulators, team_frame_positions, total_sampled, calibrated_sampled, tracker.reid


def build_analytics(accumulators, team_frame_positions, total_sampled, roster_map):
    players_out = {}
    for key, acc in accumulators.items():
        physical = compute_player_physical(acc.samples)
        if physical is None:
            continue
        team_prefix, num = key.split("#", 1)
        player_id = roster_map.get(team_prefix, {}).get(num) if roster_map else None
        out_key = player_id or key
        # Couverture = fraction du match (échantillonné) où CE joueur a pu être positionné —
        # pas fraction de ses seules apparitions à l'écran. Un joueur souvent hors champ doit
        # ressortir avec une couverture basse, pas 1.0 juste parce que les frames où on l'a vu
        # étaient toutes calibrables.
        coverage = round(len(acc.samples) / total_sampled, 3) if total_sampled else 0.0
        if coverage > 1.0:
            # Mathématiquement impossible sauf bug de regroupement (deux joueurs réels fusionnés
            # en une trace, déjà rencontré une fois) — mieux vaut le signaler bruyamment que
            # livrer silencieusement une donnée fausse.
            print(f"ATTENTION — couverture impossible ({coverage}) pour {out_key}, "
                  f"probable fusion incorrecte de deux joueurs distincts.")
            coverage = min(coverage, 1.0)
        players_out[out_key] = {
            **physical,
            "heatmapPoints": compute_heatmap_points(acc.samples),
            "visibleCoverage": coverage,
        }

    # Forme d'équipe : uniquement l'équipe "A" (la tienne, roster connu) — comme le reste du
    # schéma existant (team = ton équipe, pas l'adversaire, cf. emptyAdvancedAnalytics()).
    team = compute_team_shape(team_frame_positions.get("A", []))

    return {
        "source": "Pipeline vidéo (auto) — phase 1, portée réduite (physique, heatmap, forme d'équipe)",
        "importedAt": None,
        "players": players_out,
        "team": team,
        "passNetwork": [],
        "preciseEvents": [],
    }


if __name__ == "__main__":
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--video", required=True)
    parser.add_argument("--out", required=True)
    parser.add_argument("--sample-fps", type=float, default=5.0,
                         help="Fréquence d'échantillonnage en images/s (def. 5) — pas besoin du "
                              "25-30 im/s natif pour ces métriques.")
    parser.add_argument("--device", default="cpu", help="'cpu' (fiable partout), 'cuda:0', ou "
                         "'mps' (Apple Silicon — plus rapide en local mais couverture d'opérateurs "
                         "PyTorch parfois incomplète selon la version).")
    parser.add_argument("--max-seconds", type=float, default=None,
                         help="Limite la durée traitée, pour un test rapide sur un extrait avant "
                              "de lancer un match complet.")
    parser.add_argument("--start-seconds", type=float, default=0.0,
                         help="Démarre le traitement à ce point de la vidéo (pour un test sur un "
                              "extrait qui évite l'avant-match).")
    parser.add_argument("--roster", default=None,
                         help="Chemin vers un JSON {'A': {'<numero>': '<playerId>'}, 'B': {...}}. "
                              "Omis -> sortie en identités anonymes (étape 1a).")
    parser.add_argument("--debug-overlay", default=None,
                         help="Chemin d'une vidéo de contrôle (points suivis + mini-terrain calibré) "
                              "à générer en plus du JSON — pour le sanity-check visuel sur un extrait "
                              "court avant de lancer un match complet.")
    parser.add_argument("--checkpoint", default=None,
                         help="Chemin d'un fichier de reprise, sauvegardé régulièrement pendant le "
                              "traitement — utile pour un run long (match complet) qui risque d'être "
                              "coupé (déconnexion Colab, mise en veille de la machine). Idéalement "
                              "sur un stockage qui survit à la session (ex. Google Drive monté).")
    parser.add_argument("--checkpoint-every", type=int, default=300,
                         help="Sauvegarde le checkpoint tous les N échantillons traités (def. 300).")
    parser.add_argument("--resume", action="store_true",
                         help="Reprend depuis --checkpoint s'il existe, au lieu de repartir de zéro.")
    args = parser.parse_args()

    roster_map = None
    if args.roster:
        with open(args.roster) as fh:
            roster_map = json.load(fh)

    accumulators, team_frame_positions, total_sampled, calibrated_sampled, reid = run(
        args.video, args.sample_fps, args.device, args.max_seconds, args.debug_overlay, args.start_seconds,
        args.checkpoint, args.checkpoint_every, args.resume
    )
    # Lissage AVANT toute détection de saut implausible : la calibration recalcule chaque frame
    # indépendamment (nécessaire pour une caméra qui bouge), donc même un joueur immobile peut
    # sauter d'une frame à l'autre par simple instabilité du calage terrain, pas par confusion
    # d'identité — repéré concrètement sur un match complet (cf. discussion avec Gregory). Lisser en
    # premier réduit les faux positifs des étapes suivantes (découpage, regroupement), qui restent
    # un filet de sécurité utile pour les vraies confusions d'identité, pas le problème principal.
    for acc in accumulators.values():
        acc.samples = smooth_track_samples(acc.samples)

    n_before = len(accumulators)
    accumulators = split_implausible_tracks(accumulators)
    n_after_split = len(accumulators)
    accumulators = cluster_tracks_globally(accumulators)
    n_after_cluster = len(accumulators)
    # Filet de sécurité : le regroupement (fusions à 3+ morceaux, cas de transitivité pas encore
    # entièrement tracé) peut réintroduire un saut interne implausible même après le découpage
    # initial. Redécouper ici ne fait que séparer, jamais fusionner — sans risque, et garantit un
    # résultat final propre par construction plutôt que de dépendre d'avoir trouvé la cause exacte.
    accumulators = split_implausible_tracks(accumulators)
    analytics = build_analytics(accumulators, team_frame_positions, total_sampled, roster_map)

    with open(args.out, "w") as fh:
        json.dump(analytics, fh, ensure_ascii=False, indent=2)

    print(f"Ré-identification en flux : {reid.merges} fusion(s) sur {reid.opportunities} occasion(s) "
          f"(trace jamais vue avec un candidat récent de la même équipe disponible).")
    print(f"Découpage (traces contaminées par la ré-id en flux) : {n_before} -> {n_after_split} traces.")
    print(f"Regroupement global : {n_after_split} -> {n_after_cluster} traces.")
    print(f"Redécoupage final (filet de sécurité) : {n_after_cluster} -> {len(accumulators)} traces.")
    calib_pct = round(100 * calibrated_sampled / total_sampled) if total_sampled else 0
    print(f"OK — {len(analytics['players'])} joueur(s) suivi(s) sur {total_sampled} frames "
          f"échantillonnées ({calib_pct}% calibrées) -> {args.out}")


## Test rapide d'abord (quelques minutes)

Sur un court extrait, avant de lancer le match complet — pour repérer tout de suite un souci
d'environnement plutôt qu'après une heure d'attente.

In [ ]:
!python extract.py \
  --video "$VIDEO_PATH" \
  --out "/content/test_rapide.json" \
  --debug-overlay "/content/test_rapide_apercu.mp4" \
  --max-seconds 60 --sample-fps 2 --device cuda:0


Vérifie la vidéo de contrôle avant de continuer :

In [ ]:
from IPython.display import Video
Video("/content/test_rapide_apercu.mp4", embed=True, width=640)


## Match complet

Écrit sur ton Drive au fur et à mesure (`CHECKPOINT`, tous les 200 échantillons traités), pas
seulement à la toute fin — **si la session se déconnecte** (mise en veille de ta machine, coupure
réseau, limite Colab), il suffit de rouvrir ce notebook et de réexécuter cette cellule : `--resume`
repart automatiquement depuis le dernier checkpoint sur Drive plutôt que depuis zéro (tu dois quand
même avoir réexécuté les cellules d'installation au-dessus avant, puisque la session elle-même
repart à zéro). Ajuste `--sample-fps` si besoin (2 = plus rapide, 5 = plus précis).

In [ ]:
!python extract.py \
  --video "$VIDEO_PATH" \
  --out "$OUT_JSON" \
  --debug-overlay "$OUT_OVERLAY" \
  --checkpoint "$CHECKPOINT" --checkpoint-every 200 --resume \
  --sample-fps 2 --device cuda:0


## Et ensuite

`$OUT_JSON` s'importe directement dans Studio → rapport de match → Analyse avancée, via le bouton
"Importer des données de tracking (JSON)" déjà présent sur le site.

Pour rattacher aux vrais joueurs (`--roster`), il faut croiser manuellement quelle trace correspond
à quel numéro de maillot en regardant `$OUT_OVERLAY` (chaque joueur suivi y est étiqueté par son
identifiant de trace), puis construire un fichier `{"A": {"<id_trace>": "<player_id>"}, "B": {...}}`
à partir de l'export "numéros de maillot" du site — pas encore automatisé.